# NB4 — Official QuranMB.v2 benchmark (PER + detection F1)

Computes the numbers the IqraEval papers report, so your model can be placed
against the field (baseline F1 0.30-0.44, AraS2P winner 0.47, IQRA'26 best 0.72).

**Three phoneme sequences per clip (per the benchmark paper):**
- **canonical** = what SHOULD be said  (`IqraEval/test_references` -> `Reference_phn`)
- **annotated** = what WAS said, human-labelled  (`IqraEval/IqraEval_Test_GT`, GATED)
- **predicted** = your model's output

**Metric (paper Eq. 1-3):** align predicted vs canonical vs annotated ->
TR (mispron correctly flagged), FR (correct wrongly flagged), FA (mispron missed).
Precision=TR/(TR+FR), Recall=TR/(TR+FA), F1=2PR/(P+R).

**Access:** `QuranMB.v2` + `test_references` are public. The GT annotations are
GATED — accept terms at https://huggingface.co/datasets/IqraEval/IqraEval_Test_GT
and set an `HF_TOKEN` Kaggle secret. Without GT you still get **PER** (a valid,
comparable accuracy number); F1 needs the GT.

In [ ]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" soundfile librosa jiwer

In [ ]:
import os
import numpy as np
import torch

# HF token (needed for the gated GT). Set a Kaggle secret named HF_TOKEN after
# accepting the dataset terms; without it, only PER is computed.
try:
    from kaggle_secrets import UserSecretsClient
    _t = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _t; os.environ["HUGGING_FACE_HUB_TOKEN"] = _t
    HAVE_TOKEN = True; print("HF_TOKEN loaded.")
except Exception as e:
    HAVE_TOKEN = False; print("No HF_TOKEN (PER only, no F1):", e)

MODEL_DIR = "/kaggle/input/datasets/qasimmaajid/quran-phoneme-model/w2vbert-quran-phoneme/final"   # EDIT
SAMPLING_RATE = 16_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| model exists:", os.path.isdir(MODEL_DIR))

In [ ]:
from transformers import AutoModelForCTC, AutoFeatureExtractor, Wav2Vec2CTCTokenizer

model = AutoModelForCTC.from_pretrained(MODEL_DIR).to(DEVICE).eval()
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_DIR)
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(MODEL_DIR)
id2tok = {i: t for t, i in tokenizer.get_vocab().items()}
blank_id = tokenizer.pad_token_id
model_vocab = set(id2tok.values())
print("vocab size:", len(id2tok))

def recognise_phonemes(audio_array):
    feats = feature_extractor(np.asarray(audio_array, dtype=np.float32),
                              sampling_rate=SAMPLING_RATE, return_tensors="pt")
    feats = {k: v.to(DEVICE) for k, v in feats.items()}
    with torch.no_grad():
        ids = model(**feats).logits[0].argmax(-1).tolist()
    out, prev = [], None
    for i in ids:
        if i != prev and i != blank_id:
            sym = id2tok.get(int(i), "<unk>")
            if sym not in ("<sil>", "sil"):
                out.append(sym)
        prev = i
    return out

In [ ]:
import re as _re
# Normalise the benchmark's reference phonemes to the model's alphabet
# (strip subscript context tags, collapse doubled-subscript vowels, drop sil/<).
def _norm_one(p):
    if p in ("sil", "<sil>", "<"):
        return None
    m = _re.fullmatch(r"([A-Za-z]+)0([A-Za-z]+)0", p)
    if m and m.group(1) == m.group(2):
        base = m.group(1); return base if len(base) > 1 else base + base
    m = _re.fullmatch(r"([A-Za-z]+)[0-9]", p)
    if m:
        return m.group(1)
    return p

def normalise(seq):
    if isinstance(seq, str):
        seq = seq.split()
    return [q for q in (_norm_one(p) for p in seq) if q]

In [ ]:
# ---- Load benchmark: audio + canonical reference (+ gated GT annotation) ----
from datasets import load_dataset, Audio

qmb = load_dataset("IqraEval/QuranMB.v2", split="test")
qmb = qmb.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
print("QuranMB:", qmb)

# Canonical phoneme reference (what SHOULD be said).
ref = load_dataset("IqraEval/test_references", split="test")
print("test_references columns:", ref.column_names)
id_to_canon = {r["ID"]: r["Reference_phn"] for r in ref}

# Gated GT: columns are ['ID', 'Reference_phn', 'Annotation_phn'].
#   Reference_phn  = canonical (what should be said) -- same as test_references
#   Annotation_phn = VERBATIM human label (what was ACTUALLY said, with errors)
# We need Annotation_phn for detection F1; we also keep the GT's own
# Reference_phn so canonical and annotation are guaranteed aligned per row.
id_to_annot = None
id_to_canon_gt = {}
if HAVE_TOKEN:
    try:
        gt = load_dataset("IqraEval/IqraEval_Test_GT", split="test")
        print("GT columns:", gt.column_names)
        assert "Annotation_phn" in gt.column_names, gt.column_names
        id_to_annot    = {r["ID"]: r["Annotation_phn"] for r in gt}
        id_to_canon_gt = {r["ID"]: r["Reference_phn"]  for r in gt}
        print(f"GT loaded: {len(id_to_annot)} annotated clips (using 'Annotation_phn').")
        _ex = gt[0]
        print("  example canonical :", str(_ex["Reference_phn"])[:80])
        print("  example annotated :", str(_ex["Annotation_phn"])[:80])
    except Exception as e:
        print("GT load failed (accept terms + HF_TOKEN for F1):", e)
else:
    print("No token -> computing PER only.")

In [ ]:
# ---- Alignment helper: returns aligned (ref_sym, hyp_sym) pairs -------------
def align_pairs(ref, hyp):
    n, m = len(ref), len(hyp)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            c = 0 if ref[i-1] == hyp[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+c)
    i, j, pairs = n, m, []
    while i > 0 or j > 0:
        if i>0 and j>0 and dp[i][j]==dp[i-1][j-1]+(0 if ref[i-1]==hyp[j-1] else 1):
            pairs.append((ref[i-1], hyp[j-1])); i-=1; j-=1
        elif i>0 and dp[i][j]==dp[i-1][j]+1:
            pairs.append((ref[i-1], None)); i-=1
        else:
            pairs.append((None, hyp[j-1])); j-=1
    return pairs[::-1]

In [ ]:
# ---- Run the benchmark over all clips --------------------------------------
import jiwer

N = None   # None = all 1642 clips; set e.g. 200 for a quick check
rows = qmb if N is None else qmb.select(range(N))

pers = []
TR = FR = FA = 0   # detection counts (only filled if GT available)
n_f1 = 0

for k, ex in enumerate(rows):
    rid = ex["ID"]
    pred = recognise_phonemes(ex["audio"]["array"])
    canon = normalise(id_to_canon.get(rid, ""))
    if not canon:
        continue

    # PER vs canonical (always available)
    pers.append(jiwer.wer(" ".join(canon), " ".join(pred)))

    # Detection F1: needs the human annotation (what was actually said).
    if id_to_annot is not None and rid in id_to_annot:
        # use the GT's own canonical so canonical<->annotation align perfectly
        canon_f = normalise(id_to_canon_gt.get(rid, id_to_canon.get(rid, "")))
        annot   = normalise(id_to_annot[rid])
        ca = align_pairs(canon_f, annot)    # (canon, what was SAID)
        cp = align_pairs(canon_f, pred)     # (canon, what MODEL heard)
        said_wrong = [c != s for c, s in ca if c is not None]   # real mispron mask
        model_flag = [c != p for c, p in cp if c is not None]   # model-flag mask
        L = min(len(said_wrong), len(model_flag))
        for t in range(L):
            if said_wrong[t] and model_flag[t]:        TR += 1   # mispron caught
            elif (not said_wrong[t]) and model_flag[t]: FR += 1  # false alarm
            elif said_wrong[t] and (not model_flag[t]): FA += 1  # missed
        n_f1 += 1

    if (k+1) % 200 == 0:
        print(f"  {k+1}/{len(rows)} clips...")

print("\n================ QuranMB.v2 RESULTS ================")
print(f"clips scored (PER): {len(pers)}")
print(f"PER (phoneme error rate): {np.mean(pers):.4f}  -> phoneme accuracy {1-np.mean(pers):.1%}")
if n_f1:
    P = TR/(TR+FR) if (TR+FR) else 0.0
    R = TR/(TR+FA) if (TR+FA) else 0.0
    F1 = 2*P*R/(P+R) if (P+R) else 0.0
    print(f"\nDetection F1 (over {n_f1} annotated clips):")
    print(f"  TR={TR}  FR={FR}  FA={FA}")
    print(f"  Precision={P:.4f}  Recall={R:.4f}  F1={F1:.4f}")
    print("\nField reference: baseline F1 0.30-0.44 | AraS2P winner 0.47 | IQRA'26 best 0.72")
else:
    print("\n(No GT annotation -> F1 not computed.)")

### Notes on interpreting the result
- **PER** is directly comparable to phoneme-recognition papers (Al-Zaro 2025: 7.4%).
- **F1** is the mispronunciation-DETECTION metric the challenge ranks on.
- The F1 here is a faithful re-implementation of the paper's TR/FR/FA definition,
  but the official leaderboard uses the organisers' exact alignment/edge-cases.
  Treat this F1 as a close estimate; for an official number, submit predictions
  to the IqraEval evaluation API. The GT column auto-detect may need adjusting
  once you can see `gt.column_names` after accepting the dataset terms.